## Goal: merge two spectra with different numbers of points

This notebook showcases a few practical ways to **merge (blend)** two spectra even when they contain **different numbers of frequency points**.

You can think of each spectrum as a small table (a Pandas **DataFrame**) with frequency values and their "strength" (power/amplitude). Our goal is to create a smooth transition from one spectrum to the other, even if one spectrum has fewer peaks/points.

First, let's inspect the two example DataFrames and plot them.

In [1]:
import pandas as pd
from audiospylt import detect_dataframe_type

#load data from TSV

df1 = pd.read_csv('../tsv/speech1.tsv', delimiter='\t')
df2 = pd.read_csv('../tsv/conga3.tsv', delimiter='\t')

print("df1:", detect_dataframe_type(df1))
print("df2:", detect_dataframe_type(df2))

df1: {'type': 'single_frame', 'pairs': 41, 'freq_cols': ['Frequency (Hz)'], 'amp_cols': ['Amplitude']}
df2: {'type': 'single_frame', 'pairs': 10, 'freq_cols': ['Frequency (Hz)'], 'amp_cols': ['Amplitude']}


In [2]:
from audiospylt.multiplotter import plot_scatter

# 2D scatter for TSV peak lists.
# Expected columns: 'Frequency (Hz)', 'Amplitude'
# Tip: the axis scaling parameters mirror `py_scripts.dft_analysis.analyze_signal()`.

plot_scatter(
    dfs=[df1, df2],
    mode='markers',
    # Frequency-axis scaling:
    freq_axis_mode='linear',  # 'linear' | 'log' | 'mel' | 'mixed'
    freq_axis_mix=0.5,        # used only when freq_axis_mode='mixed'
    mixed_log_floor_hz=1.0,   # used only when freq_axis_mode='mixed'
    # Amplitude-axis scaling:
    amp_axis_mode='linear',   # 'linear' | 'log' | 'mixed'
    amp_axis_mix=0.5,         # used only when amp_axis_mode='mixed'
    amp_log_floor=1e-12,      # used for 'log'/'mixed'
    # Range handling:
    auto_plot_range=True,
)

The difference is quite large: one spectrum has many more points/peaks than the other.

To build a smooth transition, we need a rule for **pairing points from `df1` with points from `df2`**, even when the two DataFrames have different lengths. Below are a few simple approaches.

## Equal spread

**Idea:** take the DataFrame with fewer points and **duplicate (repeat) its points** until both DataFrames have the same number of rows.

The key detail is *how* we repeat them: we spread the duplicates **evenly**, so every original peak from the smaller DataFrame gets used about the same number of times.

This is simple and intuitive, but it can create "plateaus" where multiple target points come from the exact same source point.

In [3]:
from audiospylt import merge_equal_split
result_equal = merge_equal_split(df1, df2)

display(result_equal)

,freq_data1,amp_data1,freq_data2,amp_data2
0,97.368421,0.000965,31.25,0.000569
1,105.263158,0.005521,31.25,0.000569
2,110.526316,0.002423,31.25,0.000569
3,194.736842,0.001351,75.00,0.000870
4,202.631579,0.003517,75.00,0.000870
5,210.526316,0.007408,75.00,0.000870
6,215.789474,0.005467,75.00,0.000870
7,223.684211,0.004954,127.50,0.000540
8,228.947368,0.003144,127.50,0.000540
9,292.105263,0.001261,127.50,0.000540


Let's add a **time** dimension so we can visualize the merge as a *process* (a gradual transition), not just as one final mapping.

In [4]:
from audiospylt import create_time_transition_df

wave_stop_value = 15.0

new_df = create_time_transition_df(
    result_equal, 
    duration=wave_stop_value, 
    scale_amp=True
)
display(new_df)

,freq_start,freq_stop,time_start,time_stop,amp_min,amp_max
0,97.368421,31.25,0.0,15.0,0.127585,0.185020
1,105.263158,31.25,0.0,15.0,0.730145,0.185020
2,110.526316,31.25,0.0,15.0,0.320386,0.185020
3,194.736842,75.00,0.0,15.0,0.178615,0.283141
4,202.631579,75.00,0.0,15.0,0.465142,0.283141
5,210.526316,75.00,0.0,15.0,0.979685,0.283141
6,215.789474,75.00,0.0,15.0,0.722938,0.283141
7,223.684211,127.50,0.0,15.0,0.655112,0.175785
8,228.947368,127.50,0.0,15.0,0.415782,0.175785
9,292.105263,127.50,0.0,15.0,0.166797,0.175785


In [5]:
from audiospylt.multiplotter import plot_combined
from audiospylt.multiplotter import plot_combined_3d

plot_combined(dfs=[new_df])

plot_combined_3d(
    dfs=[new_df],
    axis_order=("time", "amp", "freq"),
    flip={"amp": True, "time": True},
)

## Cumulative distribution function (CDF)

We can use the **cumulative distribution function (CDF)** idea to merge spectra with *different numbers of points* in a stable way.

Instead of matching rows by "row number" (index), we match by **percentile**:

- For each DataFrame, we sort by frequency and assign every row a percentile from **0% (lowest frequency)** to **100% (highest frequency)**.
- Then we pair points that have the **same percentile**.

This naturally spreads the smaller DataFrame across the full range of the larger one.

Because the mapping is percentile-based, we can also **shape the transition**: with a bias parameter (later called `power`), we can make the merge prefer **lower** or **higher** frequencies.

Next we determine a **valid range** for the bias parameter (so that **every point** from the source DataFrame is still used at least once):

In [6]:
from audiospylt import merge_cdf, merge_cdf_analysis

power_range = merge_cdf_analysis(df1, df2, plot=True)

### How to read the CDF plot (Min Power vs Max Power)

- **x-axis (Frequency)**: the frequencies we get *after mapping* the smaller DataFrame (`freq_data_less`) onto the larger one.
- **y-axis (CDF / percentile)**: for any frequency on the x-axis, y tells you **what fraction of points are at or below** that frequency.
  - Example: y = 0.5 means "half the points are below this frequency" (the median).
- **Why there are two curves**: `Min Power` and `Max Power` are the **most extreme bias values** that still use **all** source points at least once.
  - Staying inside this range means we are *re-weighting* the merge, not *dropping* frequencies.
- **How to interpret the shape**:
  - If a curve rises **quickly** at low frequencies, **many mapped points land in the low-frequency region** (low-frequency bias).
  - If it stays **flat longer** and rises later, **more points get pushed toward higher frequencies** (high-frequency bias).

### Merge using a chosen `power`

Now we merge the two DataFrames using a specific `power` value within the valid range.

In [7]:
result_merge_cdf = merge_cdf(df1, df2, power_factor=1.5, plot=True)

new_df2 = create_time_transition_df(
    result_merge_cdf, 
    duration=15, 
    scale_amp=True
)
display(new_df2)

plot_combined(dfs=[new_df2])

plot_combined_3d(
    dfs=[new_df2],
    axis_order=("time", "amp", "freq"),
    flip={"amp": True, "time": True},
)

,freq_start,freq_stop,time_start,time_stop,amp_min,amp_max
0,97.368421,31.25,0.0,15.0,0.127585,0.185020
1,105.263158,75.00,0.0,15.0,0.730145,0.283141
2,110.526316,75.00,0.0,15.0,0.320386,0.283141
3,194.736842,127.50,0.0,15.0,0.178615,0.175785
4,202.631579,127.50,0.0,15.0,0.465142,0.175785
5,210.526316,127.50,0.0,15.0,0.979685,0.175785
6,215.789474,238.75,0.0,15.0,0.722938,1.000000
7,223.684211,238.75,0.0,15.0,0.655112,1.000000
8,228.947368,238.75,0.0,15.0,0.415782,1.000000
9,292.105263,238.75,0.0,15.0,0.166797,1.000000


By using a higher `power` value, we tilt the `df1 -> df2` merge toward **higher frequencies**, while still using **all** available frequency pairs (no points are dropped).

## Sigmoid distribution

Next we determine a **valid range** for the sigmoid factor (again, meaning we still use all points at least once):

In [8]:
from audiospylt import merge_sigmoid, merge_sigmoid_analysis

# To analyze and find the valid range (and plot)
valid_range = merge_sigmoid_analysis(df1, df2, plot=True)
print(f"Valid sigmoid range: {valid_range}")



Valid sigmoid range: [np.float64(0.01), np.float64(5.53)]


In [9]:
# To perform the merge
# merged_df = merge_sigmoid(df1, df2) # calculates mean range automatically
# OR with a specific range
result_merge_sigmoid = merge_sigmoid(df1, df2, sigmoid_range=3, plot=True)

In [10]:
new_df3 = create_time_transition_df(
    result_merge_sigmoid, 
    duration=15, 
    scale_amp=True
)
display(new_df3)

plot_combined(dfs=[new_df3])

plot_combined_3d(
    dfs=[new_df3],
    axis_order=("time", "amp", "freq"),
    flip={"amp": True, "time": True},
)

,freq_start,freq_stop,time_start,time_stop,amp_min,amp_max
0,97.368421,31.25,0.0,15.0,0.127585,0.185020
1,105.263158,75.00,0.0,15.0,0.730145,0.283141
2,110.526316,75.00,0.0,15.0,0.320386,0.283141
3,194.736842,75.00,0.0,15.0,0.178615,0.283141
4,202.631579,127.50,0.0,15.0,0.465142,0.175785
5,210.526316,127.50,0.0,15.0,0.979685,0.175785
6,215.789474,127.50,0.0,15.0,0.722938,0.175785
7,223.684211,127.50,0.0,15.0,0.655112,0.175785
8,228.947368,238.75,0.0,15.0,0.415782,1.000000
9,292.105263,238.75,0.0,15.0,0.166797,1.000000


### What if we go outside the min/max range?

If we go outside the min/max "valid range", we don't just *bias* the mapping anymore -- we start **filtering points out**.

In other words: the mapping becomes so skewed that some frequencies stop being used at all.

Let's see what happens with an intentionally very large sigmoid factor:

In [11]:
extreme_merge_sigmoid = merge_sigmoid(df1, df2, sigmoid_range=300, plot=True)

new_df4 = create_time_transition_df(
    extreme_merge_sigmoid, 
    duration=15, 
    scale_amp=True
)
display(new_df4)

plot_combined(dfs=[new_df4])

plot_combined_3d(
    dfs=[new_df4],
    axis_order=("time", "amp", "freq"),
    flip={"amp": True, "time": True},
)

,freq_start,freq_stop,time_start,time_stop,amp_min,amp_max
0,97.368421,31.25,0.0,15.0,0.127585,0.230409
1,105.263158,321.25,0.0,15.0,0.730145,1.000000
2,110.526316,321.25,0.0,15.0,0.320386,1.000000
3,194.736842,321.25,0.0,15.0,0.178615,1.000000
4,202.631579,321.25,0.0,15.0,0.465142,1.000000
5,210.526316,321.25,0.0,15.0,0.979685,1.000000
6,215.789474,321.25,0.0,15.0,0.722938,1.000000
7,223.684211,321.25,0.0,15.0,0.655112,1.000000
8,228.947368,321.25,0.0,15.0,0.415782,1.000000
9,292.105263,321.25,0.0,15.0,0.166797,1.000000


Here we get a huge bias toward **center frequencies**.

- Edge frequencies may end up with only **one** connection.
- A small number of center frequencies can end up connected to **many** others.

What happens if we use a sigmoid factor **< 0**?

In [12]:
extreme_negative_merge_sigmoid = merge_sigmoid(df1, df2, sigmoid_range=-10, plot=True)

new_df5 = create_time_transition_df(
    extreme_negative_merge_sigmoid, 
    duration=15, 
    scale_amp=True
)
display(new_df5)

plot_combined(dfs=[new_df5])

plot_combined_3d(
    dfs=[new_df5],
    axis_order=("time", "amp", "freq"),
    flip={"amp": True, "time": True},
)

,freq_start,freq_stop,time_start,time_stop,amp_min,amp_max
0,97.368421,600.00,0.0,15.0,0.127585,0.565752
1,105.263158,398.75,0.0,15.0,0.730145,0.300508
2,110.526316,398.75,0.0,15.0,0.320386,0.300508
3,194.736842,398.75,0.0,15.0,0.178615,0.300508
4,202.631579,398.75,0.0,15.0,0.465142,0.300508
5,210.526316,398.75,0.0,15.0,0.979685,0.300508
6,215.789474,393.75,0.0,15.0,0.722938,0.281394
7,223.684211,393.75,0.0,15.0,0.655112,0.281394
8,228.947368,393.75,0.0,15.0,0.415782,0.281394
9,292.105263,393.75,0.0,15.0,0.166797,0.281394


The sigmoid curve shape is the same as for a positive factor, but the **direction of the mapping flips**.

Now higher frequencies from `df1` tend to merge into lower frequencies of `df2` (and vice versa).

Because the sigmoid mapping is strongly **center-biased**, we can switch back to the CDF approach when we want a merge that favors the **edges** (very low / very high frequencies):

In [13]:
extreme_merge_cdf = merge_cdf(df1, df2, power_factor=50.5, plot=True)

new_df6 = create_time_transition_df(
    extreme_merge_cdf, 
    duration=15, 
    scale_amp=True
)
display(new_df6)

plot_combined(dfs=[new_df6])

plot_combined_3d(
    dfs=[new_df6],
    axis_order=("time", "amp", "freq"),
    flip={"amp": True, "time": True},
)

,freq_start,freq_stop,time_start,time_stop,amp_min,amp_max
0,97.368421,31.25,0.0,15.0,0.127585,0.327034
1,105.263158,546.25,0.0,15.0,0.730145,0.325985
2,110.526316,546.25,0.0,15.0,0.320386,0.325985
3,194.736842,546.25,0.0,15.0,0.178615,0.325985
4,202.631579,546.25,0.0,15.0,0.465142,0.325985
5,210.526316,546.25,0.0,15.0,0.979685,0.325985
6,215.789474,546.25,0.0,15.0,0.722938,0.325985
7,223.684211,546.25,0.0,15.0,0.655112,0.325985
8,228.947368,546.25,0.0,15.0,0.415782,0.325985
9,292.105263,546.25,0.0,15.0,0.166797,0.325985


As you can see, `df2` gets reduced to only the **two highest** and the **lowest** frequency.

Similarly, we can use a **negative** `power` factor with the CDF. Strictly speaking this is not the "standard" interpretation of a CDF, but in this notebook it's implemented as a **mirrored** mapping to flip the bias.

In [17]:
extreme_negative_merge_cdf = merge_cdf(df1, df2, power_factor=-50.5, plot=True)

new_df7 = create_time_transition_df(
    extreme_negative_merge_cdf, 
    duration=15, 
    scale_amp=True
)
display(new_df7)

plot_combined(dfs=[new_df7])

plot_combined_3d(
    dfs=[new_df7],
    axis_order=("time", "amp", "freq"),
    flip={"amp": True, "time": True},
)

,freq_start,freq_stop,time_start,time_stop,amp_min,amp_max
0,97.368421,600.00,0.0,15.0,0.127585,1.000000
1,105.263158,75.00,0.0,15.0,0.730145,0.500468
2,110.526316,75.00,0.0,15.0,0.320386,0.500468
3,194.736842,75.00,0.0,15.0,0.178615,0.500468
4,202.631579,75.00,0.0,15.0,0.465142,0.500468
5,210.526316,75.00,0.0,15.0,0.979685,0.500468
6,215.789474,75.00,0.0,15.0,0.722938,0.500468
7,223.684211,75.00,0.0,15.0,0.655112,0.500468
8,228.947368,75.00,0.0,15.0,0.415782,0.500468
9,292.105263,75.00,0.0,15.0,0.166797,0.500468


### Export

Save the result as **TSV** (including the time dimension, if present).

In [18]:
from audiospylt.io_utils import save_df_tsv

# # Save the amp/freq table (peaks_df) for reuse.
save_df_tsv(new_df7, "../tsv/merged_test.tsv")

Data saved successfully to c:\Users\egorp\Nextcloud\code\public_repos\audiospylt\tutorials_tech\../tsv/merged_test.tsv at 2026-01-31 14:53:35.279400.


'../tsv/merged_test.tsv'